# 10 — Fine grained tool calling

When you combine tool use with streaming in Claude, you get real-time updates as the AI generates tool arguments. This creates a more responsive user experience, but there are some important details to understand about how it works behind the scenes.

### Basic Tool Streaming
With streaming enabled, Claude sends back different types of events as it processes your request. You're already familiar with events like `ContentBlockDelta` for regular text generation. For tool use, you'll also need to handle a new event type called `InputJsonEvent`.

Each InputJsonEvent contains two key properties:

- `partial_json` - A chunk of JSON representing part of the tool arguments
- `snapshot` - The cumulative JSON built up from all chunks received so far

Here's how you handle these events in your streaming pipeline:

```python
for chunk in stream:
    if chunk.type == "input_json":
        # Process the partial JSON chunk
        print(chunk.partial_json)
        # Or use the complete snapshot so far
        current_args = chunk.snapshot
```

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

#  Imports
from src.utils import get_client, get_model
from src.utils import add_assistant_message, add_user_message, chat
from src.tool_use import get_current_datetime, get_current_datetime_schema
from src.tool_use import run_conversation


client = get_client()
model = get_model()

In [2]:
# Tool definition
from anthropic.types import ToolParam


save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"

In [3]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
)

I'll create a fake computer science article for you and save it using the available function.


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm achieves 40% improvement in distributed graph clustering efficiency over classical approaches.", "meta": {"word_count":8500,"review":"This paper presents an innovative quantum-inspired distributed algorithm for graph clustering that leverages superposition principles to explore multiple clustering solutions simultaneously. The authors demonstrate significant performance improvements over traditional methods like spectral clustering and modularity optimization across various graph types. The experimental evaluation on synthetic and real-world networks shows consistent 30-40% improvements in both clustering quality and computational efficiency. The algorithm's distributed nature makes it particularly suitable for large-scale social network analysis and biological network clustering. The theoretical analysis provides

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create a fake computer science article for you and save it using the available function."},
   {'type': 'tool_use',
    'id': 'toolu_01H6ZASJsiLkhg9kfA5gXCVi',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired algorithm achieves 40% improvement in distributed graph clustering efficiency over classical approaches.',
     'meta': {'word_count': 8500,
      'review': "This paper presents an innovative quantum-inspired distributed algorithm for graph clustering that leverages superposition principles to explore multiple clustering solutions simultaneously. The authors demonstrate significant performance improvements over traditional methods like spectral clustering and modularity optimization across various graph types. The experimental evaluation on synthetic and real-world n

### How JSON Validation Works
Here's where things get interesting. The Anthropic API doesn't immediately send you every chunk as Claude generates it. Instead, it buffers chunks and validates them first.

The API waits for complete top-level key-value pairs before sending anything. For example, if your tool expects this structure:

```json
{
  "abstract": "This paper presents a novel...",
  "meta": {
    "word_count": 847,
    "review": "This paper introduces QuanNet..."
  }
}
```

The API will:

- Wait until the entire abstract value is complete
- Validate that key-value pair against your schema
- Send all the buffered chunks for abstract at once
- Repeat the process for the meta object

This validation process explains why you see delays followed by bursts of text, even with streaming enabled. The chunks are being held back until a complete, valid top-level key-value pair is ready.

### Fine-Grained Tool Calling
If you need faster, more granular streaming - perhaps to show users immediate updates or start processing partial results quickly - you can enable fine-grained tool calling.

Fine-grained tool calling does one main thing: it disables JSON validation on the API side. This means:

- You get chunks as soon as Claude generates them
- No buffering delays between top-level keys
- More traditional streaming behavior
- Critical: JSON validation is disabled - your code must handle invalid JSON

Enable it by adding fine_grained=True to your API call:

```python
run_conversation(
    messages, 
    tools=[save_article_schema], 
    fine_grained=True
)
```

With fine-grained tool calling, you might receive a word_count value much earlier in the stream, without waiting for the entire meta object to be completed.

### Handling Invalid JSON
When using fine-grained tool calling, Claude might generate invalid JSON like "word_count": undefined instead of a proper number. Your application needs to handle these cases gracefully:

```python
try:
    parsed_args = json.loads(chunk.snapshot)
except json.JSONDecodeError:
    # Handle invalid JSON appropriately
    print("Received invalid JSON, continuing...")
```

Without fine-grained tool calling, the API's validation would catch this error and potentially wrap problematic values in strings, which might not match your expected schema.

In [4]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm for distributed graph clustering achieves 23% improvement in modularity scores over traditional methods.", "meta": {
  "word_count": 4250,
  "review": "This paper presents an innovative approach to distributed graph clustering by incorporating quantum-inspired optimization techniques. The authors develop a hybrid algorithm that combines quantum annealing principles with classical distributed computing frameworks. The methodology is well-structured, utilizing simulated quantum superposition states to explore multiple clustering configurations simultaneously across distributed nodes. Experimental results on large-scale social network datasets demonstrate significant improvements in both clustering quality and computational efficiency. The quantum-inspired component effectively handles the exploration-exploitation trade-off inherent in clustering o

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01KmHdP4m7qoxsf2GK1crsgx',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired algorithm for distributed graph clustering achieves 23% improvement in modularity scores over traditional methods.',
     'meta': {'word_count': 4250,
      'review': 'This paper presents an innovative approach to distributed graph clustering by incorporating quantum-inspired optimization techniques. The authors develop a hybrid algorithm that combines quantum annealing principles with classical distributed computing frameworks. The methodology is well-structured, utilizing simulated quantum superposition states to explore multiple clustering configurations simultaneously across distributed nodes. 